In [ ]:
from typing import Optional

from fastapi import APIRouter, Depends, HTTPException, status, Query
from sqlalchemy.orm import Session
from app.db import get_db
from .service import BrandProfileService
from .schemas import BrandProfileSchema
from .models import Brand
from app.rag.loader import DocumentLoader

router = APIRouter(prefix="/brand-profile", tags=["Brand Profile"])

@router.post("/mine", status_code=status.HTTP_200_OK)
async def mine_brand(
    brand_name: str,
    website_url: Optional[str] = None,
    raw_text_content: Optional[str] = None,
    db: Session = Depends(get_db)
):
    """Mines a fresh brand profile draft from raw text injection or clean url loading"""
    try:
        context_data = []
        if brand_name:
            context_data.append(f"Brand Name Target: {brand_name}")
        if website_url:
            context_data.append(f"Primary Seed URL: {website_url}")
            # Ở Phase sau khi tích hợp Crawl4AI hoặc Loader tươi:
            # loader = DocumentLoader()
            # web_docs = loader.load_file(website_url)
            # context_data.append("\n".join([d.text for d in web_docs]))
        if raw_text_content:
            context_data.append(f"Injected Guideline Content:\n{raw_text_content}")
            
        rag_context = "\n\n".join(context_data)
        
        if not raw_text_content and not website_url:
            raise HTTPException(status_code=400, detail="Mày phải cung cấp ít nhất link website hoặc nội dung text của Guideline.")
            
        # Tiến hành bóc tách thông tin trực tiếp bằng Groq JSON Mode
        mined_draft = BrandProfileService.mine_from_rag(rag_context)
        
        return {
            "status": "mined",
            "draft_profile": mined_draft,
            "message": "Kiểm tra lại dữ liệu thô trước khi lưu chính thức vào Database."
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@router.put("/{brand_id}", status_code=status.HTTP_200_OK)
async def save_profile(
    brand_id: str,
    profile: BrandProfileSchema,
    db: Session = Depends(get_db)
):
    """Saves or updates full transactional brand config profile"""
    brand = db.query(Brand).filter(Brand.id == brand_id).first()
    if not brand:
        raise HTTPException(status_code=404, detail="Không tìm thấy Brand ID tương ứng trên hệ thống.")
    
    try:
        BrandProfileService.save_profile(db, brand_id, profile.model_dump())
        return {
            "brand_id": brand_id,
            "status": "saved",
            "message": "Cấu hình thực thể Brand Voice đã được lưu thành công bằng Bulk Insert."
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@router.get("/{brand_id}", response_model=BrandProfileSchema)
async def get_profile(brand_id: str, db: Session = Depends(get_db)):
    """Gets complete full profile blueprint"""
    try:
        return BrandProfileService.get_full_profile(db, brand_id)
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@router.patch("/{brand_id}")
async def partial_update(brand_id: str, data: dict, db: Session = Depends(get_db)):
    """Patches single/partial attribute inside profile schema"""
    try:
        full = BrandProfileService.get_full_profile(db, brand_id)
        updated_data = full.model_dump()
        
        # Merge các trường dữ liệu được cập nhật cục bộ
        for k, v in data.items():
            if k in updated_data and isinstance(updated_data[k], dict) and isinstance(v, dict):
                updated_data[k].update(v)
            else:
                updated_data[k] = v
                
        BrandProfileService.save_profile(db, brand_id, updated_data)
        return {"status": "updated", "patched_fields": list(data.keys())}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@router.get("/{brand_id}/scope/writer")
async def get_writer_scope(brand_id: str, db: Session = Depends(get_db)):
    try:
        return BrandProfileService.get_writer_scope(db, brand_id).model_dump()
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

@router.get("/{brand_id}/scope/designer")
async def get_designer_scope(brand_id: str, db: Session = Depends(get_db)):
    try:
        return BrandProfileService.get_designer_scope(db, brand_id).model_dump()
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

@router.get("/{brand_id}/scope/ads")
async def get_ads_scope(brand_id: str, db: Session = Depends(get_db)):
    try:
        return BrandProfileService.get_ads_scope(db, brand_id).model_dump()
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

@router.get("/{brand_id}/scope/landing-page")
async def get_landing_page_scope(brand_id: str, db: Session = Depends(get_db)):
    try:
        return BrandProfileService.get_landing_page_scope(db, brand_id).model_dump()
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

@router.get("")
async def list_profiles(
    owner_id: str,
    limit: int = Query(10, ge=1, le=100),
    offset: int = Query(0, ge=0),
    db: Session = Depends(get_db)
):
    """Lists basic summary meta info of active owner brands"""
    try:
        base_query = db.query(Brand).filter(Brand.owner_id == owner_id, Brand.deleted_at == None)
        brands = base_query.limit(limit).offset(offset).all()
        total = base_query.count()
        
        return {
            "brands": [{"id": b.id, "name": b.name, "created_at": b.created_at} for b in brands],
            "total": total
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))